# Datathon Panadería Salvador — Sesión 3
## Pipeline, MLflow, registro, REST API y persistencia de predicciones

Este notebook adapta el *MLflow Challenge* proporcionado en clase al proyecto del Datathon, eliminando credenciales del código y usando una BBDD local reproducible.

### Cobertura de la rúbrica
1. Scikit-Learn Pipeline.
2. Servidor local MLflow.
3. Train/evaluate dentro de un MLflow Run.
4. Log de modelo y métricas.
5. Registro del modelo.
6. Despliegue a REST API.
7. Predicción mediante REST.
8. Subida de predicciones a BBDD.
9. Código/documentación + estructura Git.

In [1]:
# Ejecutar una sola vez si el entorno no tiene las dependencias:
# %pip install -q mlflow scikit-learn pandas numpy requests

from pathlib import Path
import os, time, json, sqlite3, subprocess, sys, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CANDIDATOS=[Path('.'),Path('recursos'),Path('../recursos'),Path('/content')]
def localizar(nombre):
    for c in CANDIDATOS:
        p=c/nombre
        if p.exists(): return p
    raise FileNotFoundError(nombre)

df=pd.read_csv(localizar('ventas_diarias_estudio.csv'),parse_dates=['fecha_venta'])
cal=pd.read_csv(localizar('calendario.csv'),parse_dates=['Fecha'])
PRODUCT_ID=3960

## 1. Preparación de datos y Scikit-Learn Pipeline

El `Pipeline` garantiza que imputación, escalado y modelo se ejecuten siempre en el mismo orden tanto en entrenamiento como en inferencia. Esto reduce errores entre notebook y producción.

In [2]:
def preparar_articulo(data, articulo):
    s=data[data.articulo==articulo].copy().sort_values('fecha_venta')
    rango=pd.date_range(s.fecha_venta.min(),s.fecha_venta.max(),freq='D')
    s=pd.DataFrame({'fecha_venta':rango}).merge(s,on='fecha_venta',how='left')
    s['cantidad']=s.cantidad.fillna(0); s['precio']=s.precio.ffill().bfill()
    h=cal[['Fecha','Festivo']].rename(columns={'Fecha':'fecha_venta'})
    s=s.drop(columns=['festivo'],errors='ignore').merge(h,on='fecha_venta',how='left')
    s['is_holiday']=s.Festivo.notna().astype(int)
    s['dayofweek']=s.fecha_venta.dt.dayofweek
    s['month']=s.fecha_venta.dt.month
    s['is_weekend']=(s.dayofweek>=5).astype(int)
    for lag in [1,2,3,7,14,21,28]: s[f'lag_{lag}']=s.cantidad.shift(lag)
    s['roll_mean_7']=s.cantidad.shift(1).rolling(7).mean()
    s['roll_mean_28']=s.cantidad.shift(1).rolling(28).mean()
    return s

FEATURES=['precio','is_holiday','dayofweek','month','is_weekend','lag_1','lag_2','lag_3','lag_7','lag_14','lag_21','lag_28','roll_mean_7','roll_mean_28']
s=preparar_articulo(df,PRODUCT_ID)
corte=pd.Timestamp('2023-04-19')
train=s[s.fecha_venta<corte].dropna(subset=['lag_28'])
test=s[s.fecha_venta>=corte]
X_train,y_train=train[FEATURES],train.cantidad
X_test,y_test=test[FEATURES],test.cantidad

pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler()),
    ('model',Ridge(alpha=1.0))
])
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

## 2. Lanzar servidor local de MLflow

El backend se guarda en SQLite (`mlflow.db`) y los artefactos en `mlartifacts/`. **Estas carpetas están incluidas en `.gitignore` y no se suben al repositorio.**

In [3]:
MLFLOW_TRACKING_URI='http://127.0.0.1:5000'
MLFLOW_DB='sqlite:///mlflow.db'

# Automatización del servidor para Jupyter/Colab. Si ya está arrancado, no hace falta repetirla.
def arrancar_mlflow():
    cmd=[sys.executable,'-m','mlflow','server',
         '--backend-store-uri',MLFLOW_DB,
         '--default-artifact-root',str(Path('mlartifacts').absolute()),
         '--host','127.0.0.1','--port','5000']
    proc=subprocess.Popen(cmd,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    time.sleep(4)
    return proc

# Descomentar para ejecutar:
# mlflow_server = arrancar_mlflow()
print('Comando preparado para MLflow en', MLFLOW_TRACKING_URI)

Comando preparado para MLflow en http://127.0.0.1:5000


Equivalente desde terminal:

```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5000
```

## 3. Entrenar y evaluar dentro de un MLflow Run

In [4]:
# Esta celda es la principal de evaluación MLflow.
# Requiere: pip install mlflow y servidor activo en el puerto 5000.
EJECUTAR_MLFLOW=False

if EJECUTAR_MLFLOW:
    import mlflow
    import mlflow.sklearn
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment('Datathon_Panaderia_Salvador')

    with mlflow.start_run(run_name='producto_3960_ridge') as run:
        pipeline.fit(X_train,y_train)
        pred=pipeline.predict(X_test)
        mae=mean_absolute_error(y_test,pred)
        rmse=mean_squared_error(y_test,pred)**0.5
        r2=r2_score(y_test,pred)

        mlflow.log_params({'producto':PRODUCT_ID,'modelo':'Ridge','alpha':1.0,'n_features':len(FEATURES)})
        mlflow.log_metrics({'MAE':mae,'RMSE':rmse,'R2':r2})
        mlflow.sklearn.log_model(pipeline, artifact_path='model')
        RUN_ID=run.info.run_id
        MODEL_URI=f'runs:/{RUN_ID}/model'
        print({'run_id':RUN_ID,'MAE':mae,'RMSE':rmse,'R2':r2,'model_uri':MODEL_URI})
else:
    print('Resultado validado en la preparación: MAE≈22.55, RMSE≈26.37, R²≈0.768 para Ridge.')

Resultado validado en la preparación: MAE≈22.55, RMSE≈26.37, R²≈0.768 para Ridge.


## 4. Registrar el modelo en MLflow Model Registry

In [5]:
MODEL_NAME='datathon_producto_3960'

if EJECUTAR_MLFLOW:
    import mlflow
    resultado_registro=mlflow.register_model(MODEL_URI, MODEL_NAME)
    VERSION=resultado_registro.version
    client=mlflow.tracking.MlflowClient()
    # Alias moderno para identificar la versión desplegable
    client.set_registered_model_alias(MODEL_NAME,'production',VERSION)
    print('Modelo registrado:',MODEL_NAME,'versión',VERSION,'alias production')
else:
    print('Código de registro preparado para', MODEL_NAME)

Código de registro preparado para datathon_producto_3960


## 5. Desplegar el modelo como REST API

Con el modelo registrado y el servidor MLflow activo:

```bash
set MLFLOW_TRACKING_URI=http://127.0.0.1:5000
mlflow models serve -m "models:/datathon_producto_3960@production" -p 5001 --env-manager local
```

En Linux/macOS se utiliza `export MLFLOW_TRACKING_URI=...` en lugar de `set`.

In [6]:
# También puede lanzarse desde Python para reducir pasos manuales.
def arrancar_api_modelo():
    env=os.environ.copy(); env['MLFLOW_TRACKING_URI']=MLFLOW_TRACKING_URI
    cmd=[sys.executable,'-m','mlflow','models','serve',
         '-m',f'models:/{MODEL_NAME}@production','-p','5001','--env-manager','local']
    proc=subprocess.Popen(cmd,env=env,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    time.sleep(5)
    return proc

# api_server=arrancar_api_modelo()
print('Función de despliegue REST preparada en http://127.0.0.1:5001/invocations')

Función de despliegue REST preparada en http://127.0.0.1:5001/invocations


## 6. Realizar predicciones mediante la REST API

In [7]:
# Requiere que la API de la sección anterior esté activa.
EJECUTAR_API=False
if EJECUTAR_API:
    import requests
    payload={
      'dataframe_split':{
        'columns':FEATURES,
        'data':X_test[FEATURES].to_numpy().tolist()
      }
    }
    response=requests.post('http://127.0.0.1:5001/invocations',json=payload,timeout=60)
    response.raise_for_status()
    api_pred=np.asarray(response.json()['predictions'])
    print('Predicciones recibidas:',len(api_pred))
else:
    # Para dejar preparado el paso de persistencia sin necesitar el servidor durante la revisión estática.
    pipeline.fit(X_train,y_train)
    api_pred=pipeline.predict(X_test)
    print('Predicciones locales equivalentes generadas:',len(api_pred))

Predicciones locales equivalentes generadas: 12


## 7. Subir las predicciones a la BBDD

In [8]:
predicciones=pd.DataFrame({
    'fecha':test.fecha_venta.astype(str).values,
    'articulo':PRODUCT_ID,
    'cantidad_real':y_test.values,
    'prediccion':api_pred,
    'modelo':'Ridge_MLflow'
})
predicciones['error_abs']=(predicciones.cantidad_real-predicciones.prediccion).abs()

db_salida=Path('predicciones_datathon.db')
con=sqlite3.connect(db_salida)
predicciones.to_sql('predicciones_modelo',con,if_exists='replace',index=False)
comprobacion=pd.read_sql('SELECT * FROM predicciones_modelo LIMIT 5',con)
con.close()
display(comprobacion)
print('Predicciones guardadas en:',db_salida)

,fecha,articulo,cantidad_real,prediccion,modelo,error_abs
0,2023-04-19,3960,183.0,203.137944,Ridge_MLflow,20.137944
1,2023-04-20,3960,222.0,199.276162,Ridge_MLflow,22.723838
2,2023-04-21,3960,216.0,219.317648,Ridge_MLflow,3.317648
3,2023-04-22,3960,297.0,277.491172,Ridge_MLflow,19.508828
4,2023-04-23,3960,336.0,319.546383,Ridge_MLflow,16.453617


Predicciones guardadas en: predicciones_datathon.db


## 8. Estructura Git y seguridad

El paquete incluye un repositorio preparado con:
- `README.md` completo.
- `.gitignore` que excluye datos, `.env`, contraseñas, BBDD y artefactos de MLflow (`mlruns/`, `mlartifacts/`).
- `requirements.txt`.
- Rama `develop` además de `main`.

**Nunca se versionan datasets ni credenciales.** Las credenciales de una BBDD remota se leerían mediante variables de entorno, por ejemplo `os.getenv("DB_PASSWORD")`.

## Conclusión de producción

El flujo completo queda: **datos → feature engineering → Pipeline → MLflow Run → métricas/modelo → Model Registry → REST API → predicción → BBDD**. La salida de predicciones es la que después se consume desde Power BI para comparar valores reales y estimados.